#                                                       Supply Chain Analysis 


## This project analyzes mandi price data to understand supply-demand patterns, price fluctuations, and market pressure using Python and data visualization techniques.

### Objective:

1. To analyze price variations across states and commodities.
2. To identify unstable markets using price spread.
3. To detect supply-demand imbalance using market pressure.
4. To generate meaningful insights for supply chain optimization.

### Data Collection
Dataset is collected from Government of India Open Data Platform (data.gov.in).
Dataset contains 7000+ records of mandi prices across India.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load dataset
df = pd.read_csv(r"D:\Downloads\supply_chain_data.csv")


In [ ]:
# Basic info
print("Shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())

In [ ]:
print(df.columns)
print(df.info())
print(df.describe())

In [ ]:
df.rename(columns={
    'State': 'state',
    'District': 'district',
    'Market': 'market',
    'Commodity': 'commodity',
    'Variety': 'variety',
    'Grade': 'grade',
    'Arrival_Date': 'arrival_date',
    'Min_x0020_Price': 'min_price',
    'Max_x0020_Price': 'max_price',
    'Modal_x0020_Price': 'modal_price'
}, inplace=True)

print(df.columns)

#### Data Cleaning
Cleaning the dataset is important to ensure accuracy and remove inconsistencies before analysis.

In [ ]:
print(df.isnull().sum())

df['modal_price'] = df['modal_price'].fillna(df['modal_price'].mean())
df['min_price'] = df['min_price'].fillna(0)
df['max_price'] = df['max_price'].fillna(0)

In [ ]:
# Conversions 
df['min_price'] = pd.to_numeric(df['min_price'], errors='coerce')
df['max_price'] = pd.to_numeric(df['max_price'], errors='coerce')
df['modal_price'] = pd.to_numeric(df['modal_price'], errors='coerce')

df['arrival_date'] = pd.to_datetime(df['arrival_date'], errors='coerce')

In [ ]:
df = df.drop_duplicates()

#### Feature Engineering
Created new features to measure price variation and market pressure. These features help in identifying instability and demand-supply imbalance.

In [ ]:
df['price_spread'] = df['max_price'] - df['min_price']
df['price_volatility'] = df['modal_price'] / (df['min_price'] + 1)
df['market_pressure'] = df['modal_price'] / (df['price_spread'] + 1)

Extracted date features for time-based analysis.

In [ ]:
df['year'] = df['arrival_date'].dt.year
df['month'] = df['arrival_date'].dt.month
df['day'] = df['arrival_date'].dt.day

In [ ]:
print(df.columns)

In [ ]:
df

### DATA VISUALISATION
Visualization helps in identifying patterns and trends in the dataset. Visualizing distribution of states and price patterns.

In [ ]:
import matplotlib.pyplot as plt

# Top 10 States by Data Count
plt.figure(figsize=(8,5))

df['state'].value_counts().head(10).plot(kind='bar', color='lightgreen')

plt.title("Top 10 States by Number of Records")
plt.xlabel("State")
plt.ylabel("Count")

plt.xticks(rotation=45)
plt.grid(axis='y')

plt.show()

In [ ]:
plt.hist(df['modal_price'], bins=50)
plt.title("Price Distribution")
plt.xlabel("Price")
plt.ylabel("Frequency")
plt.show()

In [ ]:
# Calculate average price per state
state_price = df.groupby('state')['modal_price'].mean().sort_values().tail(10)

plt.figure(figsize=(10,6))

bars = plt.bar(state_price.index, state_price.values)

# Add values on top of bars
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval, round(yval,2), 
             ha='center', va='bottom')

plt.title("Top 10 States with Highest Average Prices")
plt.xlabel("State")
plt.ylabel("Average Modal Price")

plt.xticks(rotation=45)
plt.grid(axis='y')

plt.show()

In [ ]:
state_price = df.groupby('state')['modal_price'].mean().sort_values()

plt.figure(figsize=(12,6))

bars = plt.bar(state_price.index, state_price.values)

# Add values on top
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval, round(yval,2), 
             ha='center', va='bottom', fontsize=8)

plt.title("Average Price Across All States")
plt.xlabel("State")
plt.ylabel("Average Modal Price")

plt.xticks(rotation=90)   # IMPORTANT (warna overlap hoga)
plt.grid(axis='y')

plt.show()

In [ ]:
# Calculate average price per commodity
commodity_price = df.groupby('commodity')['modal_price'].mean().sort_values().tail(10)

plt.figure(figsize=(12,6))

bars = plt.bar(commodity_price.index, commodity_price.values, color='orange')

# Add values on top
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval, round(yval,2), 
             ha='center', va='bottom')

plt.title("Top 10 Commodities with Highest Average Prices")
plt.xlabel("Commodity")
plt.ylabel("Average Modal Price")

plt.xticks(rotation=60)
plt.grid(axis='y')

plt.show()

In [ ]:
print("Top States by Avg Price:\n", state_price)
print("\nTop Commodities by Avg Price:\n", commodity_price)

##### ANALYSIS
Identified high pressure markets and unstable pricing regions.
Compared states based on average prices.

In [ ]:
# Identify high pressure markets(Which markets are under stress)
high_pressure = df[df['market_pressure'] > df['market_pressure'].mean()]

print(high_pressure[['state', 'market', 'commodity', 'market_pressure']].head())

In [ ]:
# Identify Unstable markets (High variation = unstable supply)
unstable = df[df['price_spread'] > df['price_spread'].mean()]

print(unstable[['state', 'market', 'commodity', 'price_spread']].head())

In [ ]:
# Compare States (Which states are costly → possible supply issues)
state_analysis = df.groupby('state')[['modal_price', 'market_pressure']].mean().sort_values(by='modal_price', ascending=False)

print(state_analysis.head())

##### Risk Classification

In [ ]:
df['risk_level'] = pd.cut(df['market_pressure'],
                         bins=[0, 5, 15, 100],
                         labels=['Low Risk', 'Medium Risk', 'High Risk'])

In [ ]:
df[['market_pressure', 'risk_level']].head(7000)

In [ ]:
# Count values
risk_counts = df['risk_level'].value_counts()

plt.figure(figsize=(8,5))

colors = ['orange', 'green', 'red']  # Medium, Low, High (logical colors)

bars = plt.bar(risk_counts.index, risk_counts.values, color=colors)

# Add values on top
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval, int(yval),
             ha='center', va='bottom', fontweight='bold')

plt.title("Risk Level Distribution in Market Data", fontsize=14)
plt.xlabel("Risk Level")
plt.ylabel("Number of Records")

plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.show()

In [ ]:
# Monthly Trend
monthly_trend = df.groupby('month')['modal_price'].mean()

monthly_trend.plot(kind='line')
plt.title("Monthly Price Trend")
plt.xlabel("Month")
plt.ylabel("Average Price")
plt.show()

Note: The dataset contains data for a single date, so time-based trend analysis is limited.
However, cross-sectional analysis across states and commodities provides meaningful insights.


### Price comparison across markets

In [ ]:
import matplotlib.pyplot as plt

commodity_name = input("Enter Commodity Name: ")

filtered_df = df[df['commodity'].str.lower() == commodity_name.lower()]

if filtered_df.empty:
    print("No data found ❌")
else:
    top_markets = filtered_df.groupby('market')['modal_price'].mean().sort_values().tail(10)

    plt.figure(figsize=(10,6))
    bars = plt.bar(top_markets.index, top_markets.values)

    # ✅ ONLY VALUES ON BARS
    for bar in bars:
        plt.text(bar.get_x() + bar.get_width()/2,
                 bar.get_height(),
                 round(bar.get_height(), 2),
                 ha='center',
                 va='bottom')

    plt.xticks(rotation=90)
    plt.title(f"{commodity_name} Price in Top Markets")
    plt.xlabel("Market")
    plt.ylabel("Price")

    plt.show()

In [ ]:
import matplotlib.pyplot as plt

commodity_name = input("Enter Commodity Name: ")

filtered_df = df[df['commodity'].str.lower() == commodity_name.lower()]

if filtered_df.empty:
    print("No data found ❌")
else:
    # 👉 State-wise average price
    state_price = filtered_df.groupby('state')['modal_price'].mean().sort_values().tail(10)

    plt.figure(figsize=(10,6))
    bars = plt.bar(state_price.index, state_price.values)

    # ✅ Values on bars
    for bar in bars:
        plt.text(bar.get_x() + bar.get_width()/2,
                 bar.get_height(),
                 round(bar.get_height(), 2),
                 ha='center',
                 va='bottom',
                 fontsize=8)

    plt.title(f"{commodity_name} Price in Top States")
    plt.xlabel("State")
    plt.ylabel("Average Price")

    plt.xticks(rotation=90)
    plt.grid(axis='y')

    plt.show()

In [ ]:
import matplotlib.pyplot as plt

commodity_name = input("Enter Commodity Name: ")
state_name = input("Enter State Name: ")

filtered_df = df[
    (df['commodity'].str.lower() == commodity_name.lower()) &
    (df['state'].str.lower() == state_name.lower())
]

if not filtered_df.empty:
    
    market_price = filtered_df.groupby('market')['modal_price'].mean().sort_values().tail(10)

    plt.figure(figsize=(10,6))
    bars = plt.bar(market_price.index, market_price.values)

    # ✅ Only values on bars
    for bar in bars:
        plt.text(bar.get_x() + bar.get_width()/2,
                 bar.get_height(),
                 round(bar.get_height(), 2),
                 ha='center',
                 va='bottom',
                 fontsize=8)

    plt.title(f"{commodity_name} Price in {state_name}")
    plt.xlabel("Market")
    plt.ylabel("Price")

    plt.xticks(rotation=90)
    plt.grid(axis='y')

    plt.show()

### 💡 Key Insights:

1. States like Delhi and Himachal Pradesh have higher average prices, which may mean that demand is high or supply is limited in these areas.

2. Some markets, like Kharar APMC, show a big difference between minimum and maximum prices, which indicates that prices are not stable there.

3. Higher market pressure values suggest that demand is more than supply, which leads to higher prices.

4. Commodities that appear frequently across markets show that they are in regular demand and are actively traded.

5. Overall, the analysis shows that prices vary across different regions, and some markets face instability due to changes in demand and supply.

In [ ]:
df.to_csv("final_supply_chain_data.csv", index=False)

### Conclusion:

In this project, we analyzed real government mandi data to understand how prices vary across different states and markets.

The analysis helped in identifying areas where prices are high, markets where prices are unstable, and situations where demand is higher than supply.

Overall, this project shows how data can be used to understand market behavior and support better decision-making in the agricultural supply chain.